# Block A: Data Discovery & Validation

Objective: Understand the exact structure, availability, and quality of the CTU-CHB dataset before any processing.

This notebook explores:
- Signal inventory (FHR, UC, sampling rates, durations)
- Outcome labels availability (pH, Apgar scores, neonatal outcomes, normal/pathological classification)
- Data quality metrics (dropouts, signal quality, sampling consistency)
- Class distribution

In [ ]:
import os
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Try to import wfdb, install if not available
try:
    import wfdb
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'wfdb'])
    import wfdb

print("Libraries imported successfully")

In [ ]:
# Define data directory
data_dir = r"C:\Users\NHaq\OneDrive - University College Cork\CTG\data\ctu-chb-intrapartum-cardiotocography-database-1.0.0"

# Verify directory exists and list files
if os.path.exists(data_dir):
    files = os.listdir(data_dir)
    hea_files = [f for f in files if f.endswith('.hea')]
    print(f"✓ Data directory found")
    print(f"✓ Total .hea files: {len(hea_files)}")
    print(f"✓ Sample files: {sorted(hea_files)[:5]}")
else:
    print(f"✗ Data directory not found: {data_dir}")

In [ ]:
def parse_hea_file(hea_path):
    """
    Parse .hea header file to extract metadata.
    Returns signal information, recording details, and outcomes.
    """
    metadata = {
        'record_name': None,
        'num_signals': 0,
        'sampling_rate': None,
        'num_samples': 0,
        'duration_seconds': 0,
        'signals': [],
        'fhr_available': False,
        'uc_available': False,
        'mhr_available': False,
        'fhr_sampling_rate': None,
        'uc_sampling_rate': None,
        'outcomes': {}
    }
    
    try:
        with open(hea_path, 'r') as f:
            lines = f.readlines()
        
        # First line: record name, num signals, sampling rate, num samples
        first_line = lines[0].strip().split()
        metadata['record_name'] = first_line[0]
        metadata['num_signals'] = int(first_line[1])
        metadata['sampling_rate'] = int(first_line[2])
        metadata['num_samples'] = int(first_line[3])
        
        # Calculate duration in seconds
        if metadata['sampling_rate'] > 0:
            metadata['duration_seconds'] = metadata['num_samples'] / metadata['sampling_rate']
        
        # Parse signal lines (one per signal)
        for i in range(metadata['num_signals']):
            if i + 1 < len(lines):
                signal_line = lines[i + 1].strip()
                # Format: file format gain baseline sample_units ADC_resolution ADC_zero initial_value checksum
                parts = signal_line.split()
                if len(parts) >= 1:
                    signal_info = {
                        'name': parts[0] if len(parts) > 0 else f'Signal_{i}',
                        'file_format': parts[0].split('.')[1] if '.' in parts[0] else 'unknown',
                        'description': ' '.join(parts[1:]) if len(parts) > 1 else ''
                    }
                    metadata['signals'].append(signal_info)
                    
                    # Check signal type
                    signal_name = signal_info['name'].lower()
                    if 'fhr' in signal_name or signal_name == 'fhr.dat':
                        metadata['fhr_available'] = True
                        metadata['fhr_sampling_rate'] = metadata['sampling_rate']
                    elif 'uc' in signal_name or signal_name == 'uc.dat':
                        metadata['uc_available'] = True
                        metadata['uc_sampling_rate'] = metadata['sampling_rate']
                    elif 'mhr' in signal_name or 'maternal' in signal_name.lower():
                        metadata['mhr_available'] = True
        
        # Parse comments for outcomes (typically start with #)
        for line in lines[metadata['num_signals'] + 1:]:
            line = line.strip()
            if line.startswith('#'):
                # Parse outcome information
                line_clean = line.lstrip('#').strip()
                if 'Outcome' in line_clean or 'outcome' in line_clean:
                    parts = line_clean.split(':')
                    if len(parts) == 2:
                        key = parts[0].strip()
                        value = parts[1].strip()
                        metadata['outcomes'][key] = value
                # Common fields in CTU-CHB dataset
                elif any(field in line_clean for field in ['pH', 'Apgar', 'SBS', 'REC TYPE']):
                    parts = line_clean.split(':')
                    if len(parts) == 2:
                        key = parts[0].strip()
                        value = parts[1].strip()
                        metadata['outcomes'][key] = value
    
    except Exception as e:
        print(f"Error parsing {hea_path}: {str(e)}")
    
    return metadata

# Test on first file
test_file = os.path.join(data_dir, sorted(hea_files)[0])
test_metadata = parse_hea_file(test_file)
print(f"\n✓ Successfully parsed: {test_file}")
print(f"  Recording: {test_metadata['record_name']}")
print(f"  Signals: {test_metadata['num_signals']}")
print(f"  Duration: {test_metadata['duration_seconds']:.1f} seconds ({test_metadata['duration_seconds']/60:.1f} minutes)")
print(f"  FHR available: {test_metadata['fhr_available']}")
print(f"  UC available: {test_metadata['uc_available']}")
print(f"  Outcomes: {test_metadata['outcomes']}")

In [ ]:
def load_signal_and_assess_quality(record_name, data_dir):
    """
    Load FHR and UC signals and assess basic quality metrics.
    """
    quality_metrics = {
        'record_name': record_name,
        'fhr_loaded': False,
        'uc_loaded': False,
        'fhr_length': 0,
        'uc_length': 0,
        'fhr_missing_pct': 0,
        'uc_missing_pct': 0,
        'fhr_outliers_pct': 0,
        'uc_outliers_pct': 0,
        'error': None
    }
    
    try:
        # Load the record using wfdb
        record_path = os.path.join(data_dir, record_name)
        record = wfdb.rdrecord(record_path)
        
        # Get signal names
        signal_names = [sig.lower() for sig in record.sig_name]
        
        # Extract FHR signal (typically first signal)
        if 'fhr' in signal_names or 'fhr.dat' in signal_names:
            fhr_idx = signal_names.index('fhr') if 'fhr' in signal_names else signal_names.index('fhr.dat')
            fhr = record.p_signal[:, fhr_idx]
            quality_metrics['fhr_loaded'] = True
            quality_metrics['fhr_length'] = len(fhr)
            
            # Check for missing values (typically marked as -32768 or NaN)
            missing_mask = (fhr == -32768) | (fhr < 0) | np.isnan(fhr)
            quality_metrics['fhr_missing_pct'] = 100 * missing_mask.sum() / len(fhr)
            
            # Check for physiologically implausible values (outliers outside [80, 240] bpm)
            valid_fhr = fhr[(fhr > 0) & (~np.isnan(fhr))]
            if len(valid_fhr) > 0:
                outlier_mask = (valid_fhr < 80) | (valid_fhr > 240)
                quality_metrics['fhr_outliers_pct'] = 100 * outlier_mask.sum() / len(valid_fhr)
        
        # Extract UC signal (typically second signal)
        if 'uc' in signal_names or 'uc.dat' in signal_names:
            uc_idx = signal_names.index('uc') if 'uc' in signal_names else signal_names.index('uc.dat')
            uc = record.p_signal[:, uc_idx]
            quality_metrics['uc_loaded'] = True
            quality_metrics['uc_length'] = len(uc)
            
            # Check for missing values
            missing_mask = (uc == -32768) | (uc < 0) | np.isnan(uc)
            quality_metrics['uc_missing_pct'] = 100 * missing_mask.sum() / len(uc)
            
            # UC typically has outliers from baseline drift
            valid_uc = uc[(uc > 0) & (~np.isnan(uc))]
            if len(valid_uc) > 0:
                outlier_mask = (valid_uc > 150)  # UC baseline typically < 150
                quality_metrics['uc_outliers_pct'] = 100 * outlier_mask.sum() / len(valid_uc)
    
    except Exception as e:
        quality_metrics['error'] = str(e)
    
    return quality_metrics

# Test on first few files
print("Testing signal loading and quality assessment...")
test_quality = load_signal_and_assess_quality(sorted(hea_files)[0].replace('.hea', ''), data_dir)
print(f"  FHR Loaded: {test_quality['fhr_loaded']}, Length: {test_quality['fhr_length']}")
print(f"  UC Loaded: {test_quality['uc_loaded']}, Length: {test_quality['uc_length']}")
print(f"  FHR Missing: {test_quality['fhr_missing_pct']:.2f}%")
print(f"  UC Missing: {test_quality['uc_missing_pct']:.2f}%")

In [ ]:
def extract_outcome_label(outcomes_dict):
    """
    Extract the primary outcome label (normal vs pathological) from outcomes dictionary.
    Common fields: 'REC TYPE', 'Outcome', 'outcome'
    """
    label = None
    label_field = None
    
    for key, value in outcomes_dict.items():
        value_lower = str(value).lower().strip()
        # Look for REC TYPE field (typically contains N or P)
        if 'rec type' in key.lower():
            label_field = key
            if 'n' in value_lower or 'normal' in value_lower:
                label = 'Normal'
            elif 'p' in value_lower or 'path' in value_lower:
                label = 'Pathological'
        # Look for explicit outcome field
        elif 'outcome' in key.lower():
            label_field = key
            if 'normal' in value_lower:
                label = 'Normal'
            elif 'path' in value_lower or 'abnormal' in value_lower:
                label = 'Pathological'
    
    return label, label_field

# Test extraction
print("Testing outcome extraction...")
test_label, test_field = extract_outcome_label(test_metadata['outcomes'])
print(f"  Outcome label: {test_label}")
print(f"  Field: {test_field}")
print(f"  Raw outcomes: {test_metadata['outcomes']}")

In [ ]:
print("=" * 80)
print("BLOCK A: DATA DISCOVERY & VALIDATION - SCANNING ALL RECORDS")
print("=" * 80)

# Initialize collection lists
all_records = []
signal_inventory = defaultdict(int)
outcome_fields = defaultdict(set)

# Scan all .hea files
print(f"\nScanning {len(hea_files)} recordings...\n")

for idx, hea_file in enumerate(sorted(hea_files)):
    if idx % 100 == 0:
        print(f"Progress: {idx}/{len(hea_files)}")
    
    record_name = hea_file.replace('.hea', '')
    hea_path = os.path.join(data_dir, hea_file)
    
    # Parse header
    metadata = parse_hea_file(hea_path)
    
    # Load and assess signal quality
    quality = load_signal_and_assess_quality(record_name, data_dir)
    
    # Extract outcome label
    outcome_label, outcome_field = extract_outcome_label(metadata['outcomes'])
    
    # Collect signal types
    signal_inventory['Total'] += 1
    if metadata['fhr_available']:
        signal_inventory['FHR'] += 1
    if metadata['uc_available']:
        signal_inventory['UC'] += 1
    if metadata['mhr_available']:
        signal_inventory['MHR'] += 1
    
    # Collect outcome fields
    for key in metadata['outcomes'].keys():
        outcome_fields[key].add(True)
    
    # Compile record info
    record_info = {
        'record_id': record_name,
        'num_signals': metadata['num_signals'],
        'sampling_rate': metadata['sampling_rate'],
        'duration_minutes': metadata['duration_seconds'] / 60,
        'fhr_available': metadata['fhr_available'],
        'uc_available': metadata['uc_available'],
        'mhr_available': metadata['mhr_available'],
        'fhr_missing_pct': quality['fhr_missing_pct'],
        'uc_missing_pct': quality['uc_missing_pct'],
        'fhr_outliers_pct': quality['fhr_outliers_pct'],
        'uc_outliers_pct': quality['uc_outliers_pct'],
        'outcome_label': outcome_label,
        **metadata['outcomes']  # Include all outcome fields
    }
    
    all_records.append(record_info)

print(f"\n✓ Successfully scanned {len(all_records)} recordings\n")

In [ ]:
# Create DataFrame for easier analysis
df = pd.DataFrame(all_records)

print("=" * 80)
print("SIGNAL INVENTORY")
print("=" * 80)
print(f"\nSignal Availability:")
for signal_type, count in sorted(signal_inventory.items()):
    pct = 100 * count / signal_inventory['Total']
    print(f"  {signal_type:20s}: {count:4d} / {signal_inventory['Total']} ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("RECORDING DURATION STATISTICS")
print("=" * 80)
print(f"\nDuration (minutes):")
print(f"  Mean:     {df['duration_minutes'].mean():.2f}")
print(f"  Median:   {df['duration_minutes'].median():.2f}")
print(f"  Std Dev:  {df['duration_minutes'].std():.2f}")
print(f"  Min:      {df['duration_minutes'].min():.2f}")
print(f"  Max:      {df['duration_minutes'].max():.2f}")
print(f"  Total:    {df['duration_minutes'].sum():.0f} minutes ({df['duration_minutes'].sum()/60:.1f} hours)")

print("\n" + "=" * 80)
print("OUTCOME LABEL DISTRIBUTION")
print("=" * 80)
outcome_counts = df['outcome_label'].value_counts()
print(f"\nClass Distribution:")
for outcome, count in outcome_counts.items():
    if pd.notna(outcome):
        pct = 100 * count / len(df)
        print(f"  {outcome:20s}: {count:4d} ({pct:5.1f}%)")

unknown_count = df['outcome_label'].isna().sum()
if unknown_count > 0:
    pct = 100 * unknown_count / len(df)
    print(f"  {'Unknown/Missing':20s}: {unknown_count:4d} ({pct:5.1f}%)")

In [ ]:
print("\n" + "=" * 80)
print("SIGNAL QUALITY METRICS")
print("=" * 80)

print(f"\nFHR Missing Data (%):")
print(f"  Mean:     {df['fhr_missing_pct'].mean():.2f}%")
print(f"  Median:   {df['fhr_missing_pct'].median():.2f}%")
print(f"  Max:      {df['fhr_missing_pct'].max():.2f}%")
print(f"  Records with <50% valid data: {(df['fhr_missing_pct'] > 50).sum()}")
print(f"  Records with >50% valid data: {(df['fhr_missing_pct'] <= 50).sum()}")

print(f"\nFHR Outliers (outside [80, 240] bpm):")
print(f"  Mean:     {df['fhr_outliers_pct'].mean():.2f}%")
print(f"  Median:   {df['fhr_outliers_pct'].median():.2f}%")
print(f"  Max:      {df['fhr_outliers_pct'].max():.2f}%")

print(f"\nUC Missing Data (%):")
print(f"  Mean:     {df['uc_missing_pct'].mean():.2f}%")
print(f"  Median:   {df['uc_missing_pct'].median():.2f}%")
print(f"  Max:      {df['uc_missing_pct'].max():.2f}%")

print(f"\nUC Outliers:")
print(f"  Mean:     {df['uc_outliers_pct'].mean():.2f}%")
print(f"  Median:   {df['uc_outliers_pct'].median():.2f}%")
print(f"  Max:      {df['uc_outliers_pct'].max():.2f}%")

print("\n" + "=" * 80)
print("OUTCOME FIELDS AVAILABLE IN DATASET")
print("=" * 80)
print(f"\nFields captured from header comments:")
for field in sorted(outcome_fields.keys()):
    count = len(outcome_fields[field])
    print(f"  {field:30s}: Found in records")

print("\n" + "=" * 80)
print("DATA QUALITY INCLUSION CRITERIA (≥50% valid FHR in 30-min windows)")
print("=" * 80)
quality_pass = (df['fhr_missing_pct'] <= 50).sum()
quality_fail = (df['fhr_missing_pct'] > 50).sum()
print(f"\nRecords passing inclusion criteria: {quality_pass}/{len(df)} ({100*quality_pass/len(df):.1f}%)")
print(f"Records failing inclusion criteria: {quality_fail}/{len(df)} ({100*quality_fail/len(df):.1f}%)")

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS TABLE")
print("=" * 80)

summary_stats = {
    'Metric': [
        'Total recordings analyzed',
        'Mean recording duration (minutes)',
        'Median recording duration (minutes)',
        'Recordings with FHR signal',
        'Recordings with UC signal',
        'Recordings with MHR signal',
        'Normal cases',
        'Pathological cases',
        'Unknown/unlabeled cases',
        'Records passing quality criteria (≥50% valid FHR)',
        'Records failing quality criteria (<50% valid FHR)',
        'Mean FHR missing data (%)',
        'Mean FHR outliers (%)',
        'Mean UC missing data (%)',
        'Mean UC outliers (%)'
    ],
    'Value': [
        len(df),
        f"{df['duration_minutes'].mean():.2f}",
        f"{df['duration_minutes'].median():.2f}",
        f"{df['fhr_available'].sum()}",
        f"{df['uc_available'].sum()}",
        f"{df['mhr_available'].sum()}",
        f"{(df['outcome_label'] == 'Normal').sum()}",
        f"{(df['outcome_label'] == 'Pathological').sum()}",
        f"{df['outcome_label'].isna().sum()}",
        f"{quality_pass}",
        f"{quality_fail}",
        f"{df['fhr_missing_pct'].mean():.2f}%",
        f"{df['fhr_outliers_pct'].mean():.2f}%",
        f"{df['uc_missing_pct'].mean():.2f}%",
        f"{df['uc_outliers_pct'].mean():.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_stats)
print("\n" + summary_df.to_string(index=False))

print("\n" + "=" * 80)

In [ ]:
# Save summary report to CSV
output_dir = r"C:\Users\NHaq\OneDrive - University College Cork\CTG\outputs"
os.makedirs(output_dir, exist_ok=True)

# Full dataset summary
csv_path = os.path.join(output_dir, 'dataset_summary.csv')
df.to_csv(csv_path, index=False)
print(f"✓ Full dataset summary saved to: {csv_path}")

# Summary statistics
json_path = os.path.join(output_dir, 'dataset_statistics.json')
json_summary = {
    'total_recordings': int(len(df)),
    'mean_duration_minutes': float(df['duration_minutes'].mean()),
    'median_duration_minutes': float(df['duration_minutes'].median()),
    'signal_inventory': {
        'fhr_available': int(df['fhr_available'].sum()),
        'uc_available': int(df['uc_available'].sum()),
        'mhr_available': int(df['mhr_available'].sum())
    },
    'class_distribution': {
        'normal': int((df['outcome_label'] == 'Normal').sum()),
        'pathological': int((df['outcome_label'] == 'Pathological').sum()),
        'unknown': int(df['outcome_label'].isna().sum())
    },
    'quality_metrics': {
        'fhr_missing_pct_mean': float(df['fhr_missing_pct'].mean()),
        'fhr_outliers_pct_mean': float(df['fhr_outliers_pct'].mean()),
        'uc_missing_pct_mean': float(df['uc_missing_pct'].mean()),
        'uc_outliers_pct_mean': float(df['uc_outliers_pct'].mean()),
        'records_passing_criteria': int(quality_pass),
        'records_failing_criteria': int(quality_fail)
    }
}

with open(json_path, 'w') as f:
    json.dump(json_summary, f, indent=2)
print(f"✓ Dataset statistics saved to: {json_path}")

print("\nBlock A Complete: Data Discovery & Validation ✓")